In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# ==========================================
# STEP 1: CREATE DATASET
# ==========================================

rng = np.random.RandomState(0)

n = 200

df = pd.DataFrame({
    "age": rng.normal(40, 10, n),
    "income": rng.normal(50000, 15000, n),
    "city": rng.choice(["NY", "LA", "SF"], n)
})

city_bonus = df["city"].map({
    "SF": 1.0,
    "LA": 0.0,
    "NY": -1.0
}).values

score = (
    (df["income"].values - 50000.0) / 15000.0
    + city_bonus
)

y = (score > 0).astype(int)


# ==========================================
# STEP 2: STANDARDIZATION FUNCTIONS
# ==========================================

def standardize_fit(X):
    X = np.asarray(X, dtype=float)

    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)

    # Handle zero variance columns
    std[std == 0] = 1.0

    return mean, std


def standardize_transform(X, mean, std):
    X = np.asarray(X, dtype=float)

    return (X - mean) / std


# ==========================================
# STEP 3: ONE HOT ENCODING
# ==========================================

def one_hot(labels, categories):
    labels = np.asarray(labels)
    categories = np.asarray(categories)

    result = np.zeros(
        (len(labels), len(categories)),
        dtype=float
    )

    for i, label in enumerate(labels):
        matches = np.where(categories == label)[0]

        if len(matches) > 0:
            result[i, matches[0]] = 1.0

    return result


# ==========================================
# STEP 4: BUILD PIPELINE
# ==========================================

def build_pipeline(numeric_cols, categorical_cols):

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_cols
            )
        ]
    )

    pipeline = Pipeline([
        ("pre", preprocessor),
        (
            "model",
            LogisticRegression(max_iter=5000)
        )
    ])

    return pipeline


# ==========================================
# STEP 5: VERIFY STANDARDIZATION
# ==========================================

X = np.array([
    [1., 100., 5.],
    [3., 300., 5.],
    [5., 500., 5.],
    [7., 700., 5.]
])

m, s = standardize_fit(X)

print("Mean:", m)
print("Standard Deviation:", s)

Z = standardize_transform(X, m, s)

print("Standardized Data:\n", Z)

print("Column Mean:", Z[:, :2].mean(0))
print("Column Std:", Z[:, :2].std(0))


# ==========================================
# STEP 6: VERIFY ONE HOT ENCODING
# ==========================================

encoded = one_hot(
    ["a", "b", "c", "a"],
    ["a", "b", "c"]
)

print("\nOne Hot Encoded Data:")
print(encoded)


# ==========================================
# STEP 7: TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    df,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ==========================================
# STEP 8: BUILD AND TRAIN PIPELINE
# ==========================================

pipe = build_pipeline(
    numeric_cols=["age", "income"],
    categorical_cols=["city"]
)

pipe.fit(X_train, y_train)


# ==========================================
# STEP 9: PREDICTION
# ==========================================

y_pred = pipe.predict(X_test)

print("\nPredictions:")
print(y_pred)


# ==========================================
# STEP 10: CHECK PIPELINE
# ==========================================

print("\nPipeline Steps:")
print(pipe.named_steps)

print(
    "\nNumber of transformed features:",
    pipe.named_steps["pre"].transform(X_train).shape[1]
)


# ==========================================
# STEP 11: VERIFY UNSEEN DATA
# ==========================================

unseen_data = pd.DataFrame({
    "age": [35, 45],
    "income": [60000, 40000],
    "city": ["SF", "LA"]
})

unseen_transformed = pipe.named_steps["pre"].transform(
    unseen_data
)

print("\nUnseen Data Transformation:")
print(unseen_transformed)

print("\nUnseen Data Predictions:")
print(pipe.predict(unseen_data))


# ==========================================
# STEP 12: MODEL ACCURACY
# ==========================================

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)
